In [74]:
import numpy as np
import cmath as cm

# ESTUDO DE PROTEÇÃO E SELETIVIDADE

Abaixo apresentamos o estudo de proteção e seletividade na média tensão classe 25KV, para o consumidor Semasa localizada na cidade de Itajaí SC, conforme segue:

Dados fornecidos pela Cia Concessionária Celesc, localizada junto ao ponto de entrega, doravante denominado dados do Sistema:

### Impedâncias da concessionária

In [75]:
Z0 = 0.72 + 1j*3.74
Z1 = 0.44 + 1j*0.93

### Grandezas Base

In [76]:
Sb = 100e6
Vb = 23e3

### Correntes de curtos-circuitos

In [77]:
Icc3fA_entrada = 2636
Icc3f_entrada = 2444
Icc1fA_entrada = 1520
Icc1f_entrada = 1293
Icc1fm_entrada = 701

### Cálculo da Impedância Base

In [78]:
Zb = Vb**2/Sb
Zb

### Cálculo da corrente base

In [79]:
Ib = Sb / ( np.sqrt(3) * Vb )
Ib

## 02 - Cálculo das correntes de curto circuito na barra da cabina de medição:

Para o cálculo das correntes de curto-circuito no barramento geral da cabina de
medição/proteção, temos que considerar a impedância total do ponto de entrega até a cabina de medição, conforme segue abaixo:

### Impedância Equivalente da Concessionária;

In [80]:
print("Z0 = ",Z0)
print("Z1 = ",Z1)

Z0 =  (0.72+3.74j)
Z1 =  (0.44+0.93j)


### Impedância dos Cabos do Ramal de Entrada

Conforme esquema unifilar geral, a edificação será alimentada através de um ramal aéreo, composto por cabos de alumínio nú secção nominal 2/0AWG, numa distância de 27m, entre o ponto de entrega até a subestação. 

Conforme dados de impedâncias em Ω/Km fornecidas pelo fabricante dos cabos, calculamos o valor das impedâncias na base do sistema:


### Cálculo impedância do ramal de entrada:


In [81]:
comp = 27
Z1_cabo = ( 0.7152 + 1j*0.3871 ) * 1e-3
Z0_cabo = ( 1.8792 + 1j*1.6976 ) * 1e-3
Z1_re = comp * Z1_cabo / Zb
Z0_re = comp * Z0_cabo / Zb
print("Z1_re = ","{:3.6f}".format(Z1_re.real)," +  \u2149","{:3.6f}".format(Z1_re.imag),"   [pu]")
print("Z0_re = ","{:3.6f}".format(Z0_re.real)," +  \u2149","{:3.6f}".format(Z0_re.imag),"   [pu]")

Z1_re =  0.003650  +  ⅉ 0.001976    [pu]
Z0_re =  0.009591  +  ⅉ 0.008664    [pu]


In [82]:
Z1_total = Z1 + Z1_re
Z0_total = Z0 + Z0_re

### Corrente de Curto Trifásica Simétrica

In [83]:
Icc3f_pu = 1 / Z1_total
Icc3f = Icc3f_pu * Ib
Icc3f_polar = cm.polar(Icc3f)
print("Icc3f = ","{:6.6f}".format(Icc3f.real)," +  \u2149","{:6.6f}".format(Icc3f.imag),"   [A]")
print("Icc3f = ","{:6.6f}".format(Icc3f_polar[0]),"   \u2220","{:6.6f}".format(180/cm.pi*Icc3f_polar[1]),"   [A]")

Icc3f =  1045.292596  +  ⅉ -2195.844831    [A]
Icc3f =  2431.948012    ∠ -64.544053    [A]


### Corrente de Curto Assimétrica

Para o cálculo da corrente de curto-circuito assimétrica calcularemos o fator de assimétrica, que é dado por:

$$FA = \sqrt {1 + 2{e^{\frac{{ - \pi }}{{\left( {X/R} \right)}}}}} $$

In [84]:
R = Z1_total.real
X = Z1_total.imag
expoente = - 2* np.pi / ( X/R ) ####### aqui não bate com o MAMEDE, precisa tirar o 2 por certo
FA = np.sqrt(1+2*np.exp(expoente))
print('X/R = {:3.3f}'.format(X/R))
print("FA =","{:3.3f}".format(FA))
Icc3fass = FA * Icc3f
Icc3fass_polar = cm.polar(Icc3fass)
print("Icc3fass = ","{:6.6f}".format(Icc3fass_polar[0]),"   \u2220","{:6.6f}".format(180/cm.pi*Icc3fass_polar[1]),"   [A]")

X/R = 2.101
FA = 1.049
Icc3fass =  2551.198742    ∠ -64.544053    [A]


### Corrente de Curto Bifásica

$${I_{cc2\phi }} = \frac{{\sqrt 3 }}{2}{I_{cc3\phi }}$$

In [85]:
Icc2f = np.sqrt(3)/2 * Icc3fass
Icc2f_polar = cm.polar(Icc2f)
print("Icc2f = ","{:6.6f}".format(Icc2f_polar[0]),"   \u2220","{:6.6f}".format(180/cm.pi*Icc2f_polar[1]),"   [A]")

Icc2f =  2209.402921    ∠ -64.544053    [A]


### Corrente de Curto Fase-Terra

$${I_{cc1\phi }} = 3\left( {\frac{1}{{2{Z_1} + {Z_0}}}} \right){I_b}$$

In [86]:
denominador = 2*Z1_total + Z0_total
Icc1f = 3/denominador * Ib
Icc1f_polar = cm.polar(Icc1f)
print("Icc1f = ","{:6.6f}".format(Icc1f_polar[0]),"   \u2220","{:6.6f}".format(180/cm.pi*Icc1f_polar[1]),"   [A]")

Icc1f =  1289.303086    ∠ -73.929276    [A]


### Cálculo de Curto Monofásico Mínima

A corrente de curto circuito monofásica mínima é calculada levando em consideração a impedância de sequência positiva, zero e a resistência de terra (Rt) no ponto e é calculada com segue:

$${Z_{rt}}\left[ {{\rm{pu}}} \right] = \frac{{{R_t}\left[ \Omega  \right]}}{{{Z_b}\left[ \Omega  \right]}}$$



In [87]:
Rt = 21 * Zb ####### Precisa ser definida, aqui adapatei um valor
Zrt = Rt / Zb
print("Rt = {:3.3}".format(Rt),"  [\u03A9]")
print("Zrt = {:3.3}".format(Zrt),"  [pu]")

Rt = 1.11e+02   [Ω]
Zrt = 21.0   [pu]



$${I_{cc1f\min }} = 3\left( {\frac{1}{{2{Z_1} + \left[ {{Z_0} + {Z_{rt}}} \right]}}} \right){I_b}$$

In [88]:
denominador = 2*Z1_total + Z0_total + Zrt
Icc1fmin = 3/denominador * Ib
Icc1fmin_polar = cm.polar(Icc1fmin)
print("Icc1fmin = ","{:6.6f}".format(Icc1fmin_polar[0]),"   \u2220","{:6.6f}".format(180/cm.pi*Icc1fmin_polar[1]),"   [A]")

Icc1fmin =  323.163849    ∠ -13.936997    [A]


### Transformador de Corrente de PROTEÇÃO



*   A corrente nominal primária deverá se maior que a corrente de carga.
*   Não saturar com a maior Icc trifásica assimétrica.

#### Cálculo da tensão de sasturação

Dados:
* impedância do relé
* potência do relé
* impedância do TC (Tabela 10 da NBR 6856, considerando 20% de Carga)
* impedância dos cabos de interligação


In [89]:
Z_rele = 10e-3
S_rele = 0.25
Z_TC = 0.1

L = 4
S = 2.5
Z_cabosInterligacao = 0.02 * L / S

Z_total_TC = Z_cabosInterligacao + Z_rele + Z_TC
print("Z_total_TC = {:6.3f}".format(1e3*Z_total_TC),"   [m\u03A9]")

Z_total_TC = 142.000    [mΩ]


desta forma, o maior curto-cricuito no TC é:

In [90]:
max_cor_curto_entrada = np.amax([Icc3fA_entrada,Icc3f_entrada,Icc1fA_entrada,Icc1f_entrada])
Icc_max_TC = Icc3fA_entrada / 20
print("Icc_max_TC = {:3.0f}".format(Icc_max_TC),"   [A]")

Icc_max_TC = 132    [A]


In [91]:
Vsat_TC = Icc_max_TC * Z_total_TC 
Vsat_TC
print("Vsat_TC = {:3.0f}".format(Vsat_TC),"   [V]")

Vsat_TC =  19    [V]


Agora é buscar o TC adequado em uma tabela.

No caso o TC escolhido terá corrente nominal de $50 \, \rm{A}$ e clase de exatidão $\rm{10B50}$.

$$\rm{TC} \; 50/5 \;  \; \rm{10B50}$$

### Cálculo do Inrush dos transformadores

A Iinrush é a corrente de magnetização do transformador, conforme orientação Celesc, estamos considerando a corrente inrush como sendo igual a $8 \times I_n$ para o maior transformador da instalação, com tempo de duração da ordem de $0,1 \, \rm{s}$.

Assim:

In [92]:
In_maiorTrafo = 750e3/(np.sqrt(3)*Vb)
Inrush_maiorTrafo = 8 * In_maiorTrafo
print("Inrush_maiorTrafo = {:3.0f}".format(Inrush_maiorTrafo),"  [A]")

In_maiorTrafo = 151   [A]


## Cálculo de ajuste do relé



### Ajustes de Fase Temporizado - 51

Tomando como base a demanda declarada/solicitada junto a Celesc:

In [93]:
demandaContratada = 560e3
Ic = demandaContratada/( np.sqrt(3)*Vb*0.92 )
print("Corrente para a demanda contrada = {:3.1f}".format(Ic),"  [A]")

Corrente para a demanda contrada = 15.3   [A]


A corrente da função 51 do relé será ajustada em $1.1$ vezes este valor:

In [101]:
In51f = 1.1 * Ic
print("In51f =  {:3.0f}".format(In51f),"  [A]")

In51f =   17   [A]


### Ajuste de Fase Instantâneo - 50

A sobrecorrente instantânea de fase deverá atender a seguinte condição:

$${\rm{Inrush}} \le {I_{50F}} \le {I_{cc3\phi }}$$

Lembrando que ${\rm{Inrush}}$ e ${I_{cc3\phi }}$ são:


In [102]:
print("Inrush_maiorTrafo = {:3.0f}".format(Inrush_maiorTrafo),"  [A]")
print("Icc3fA_entrada = {:3.0f}".format(Icc3fA_entrada),"  [A]")

Inrush_maiorTrafo = 151   [A]
Icc3fA_entrada = 2636   [A]


Assim adota-se ${I_{n50F}}$ como 

In [103]:
In50f = 600
print("In50f = {:3.0f}".format(In50f),"  [A]")

In50f = 600   [A]


### Ajustes de neutro

Para o ajuste temporizado de neutro adota-se a corrente de ajuste como $20\%$ do ajuste de fase.

Assim:


In [106]:
In51n = np.ceil(0.20 * In51f) 
print("In51n =  {:3.1f}".format(In51n),"  [A]")

In51n =  4.0   [A]


Ajuste de Neutro Instantâneo - 

In [109]:
In50n = np.ceil(0.20 * In50f) 
print("In50n =  {:3.1f}".format(In50n),"  [A]")

In50n =  120.0   [A]
